# Python Decorators Cheatsheet

Modify or extend function/class behavior without changing source code.

## 1. Decorator Basics

In [ ]:
# Functions are first-class objects
def greet(name):
    return f"Hello, {name}!"

# Assign function to variable
say_hello = greet
print(say_hello("Alice"))

# Pass function as argument
def execute(func, arg):
    return func(arg)

print(execute(greet, "Bob"))

In [ ]:
# Simple decorator
def my_decorator(func):
    def wrapper():
        print("Before function call")
        func()
        print("After function call")
    return wrapper

# Apply decorator manually
def say_hello():
    print("Hello!")

decorated = my_decorator(say_hello)
decorated()

In [ ]:
# Using @ syntax
def my_decorator(func):
    def wrapper():
        print("Before function call")
        func()
        print("After function call")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()  # Automatically wrapped

## 2. Decorators with Arguments

In [ ]:
# Handle function arguments with *args and **kwargs
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Finished {func.__name__}")
        return result
    return wrapper

@my_decorator
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

print(greet("Alice"))
print(greet("Bob", greeting="Hi"))

In [ ]:
# Preserving function metadata with functools.wraps
from functools import wraps

def my_decorator(func):
    @wraps(func)  # Preserves __name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def greet(name):
    """Greet someone."""
    return f"Hello, {name}!"

print(f"Name: {greet.__name__}")
print(f"Doc: {greet.__doc__}")

## 3. Decorators with Parameters

In [ ]:
# Decorator that takes arguments
def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def say_hello(name):
    print(f"Hello, {name}!")

say_hello("Alice")

In [ ]:
# Decorator with optional arguments
def debug(func=None, prefix="DEBUG"):
    def decorator(f):
        @wraps(f)
        def wrapper(*args, **kwargs):
            print(f"{prefix}: Calling {f.__name__}")
            return f(*args, **kwargs)
        return wrapper
    
    if func is not None:
        return decorator(func)
    return decorator

@debug  # Without parentheses
def greet1():
    print("Hello!")

@debug(prefix="INFO")  # With argument
def greet2():
    print("Hello!")

greet1()
greet2()

## 4. Common Decorator Patterns

In [ ]:
# Timing decorator
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper

@timer
def slow_function():
    time.sleep(0.1)
    return "Done"

slow_function()

In [ ]:
# Logging decorator
def logger(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper

@logger
def add(a, b):
    return a + b

add(3, 5)

In [ ]:
# Caching/Memoization decorator
def memoize(func):
    cache = {}
    @wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    return wrapper

@memoize
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

print(f"fibonacci(30) = {fibonacci(30)}")

# Or use built-in
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

In [ ]:
# Retry decorator
import random

def retry(max_attempts=3, delay=1):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    attempts += 1
                    print(f"Attempt {attempts} failed: {e}")
                    if attempts < max_attempts:
                        time.sleep(delay)
            raise Exception(f"Failed after {max_attempts} attempts")
        return wrapper
    return decorator

@retry(max_attempts=3, delay=0.1)
def unreliable_function():
    if random.random() < 0.7:
        raise ValueError("Random failure!")
    return "Success!"

try:
    print(unreliable_function())
except Exception as e:
    print(e)

In [ ]:
# Validation decorator
def validate_positive(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        for arg in args:
            if isinstance(arg, (int, float)) and arg < 0:
                raise ValueError(f"Arguments must be positive, got {arg}")
        return func(*args, **kwargs)
    return wrapper

@validate_positive
def calculate_area(width, height):
    return width * height

print(calculate_area(5, 3))

try:
    print(calculate_area(-5, 3))
except ValueError as e:
    print(f"Error: {e}")

## 5. Chaining Decorators

In [ ]:
# Multiple decorators are applied bottom-up
def bold(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italic(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

@bold
@italic  # Applied first
def greet(name):
    return f"Hello, {name}!"

print(greet("Alice"))  # <b><i>Hello, Alice!</i></b>

## 6. Class Decorators

In [ ]:
# Decorator as a class
class Timer:
    def __init__(self, func):
        self.func = func
        wraps(func)(self)
    
    def __call__(self, *args, **kwargs):
        start = time.time()
        result = self.func(*args, **kwargs)
        end = time.time()
        print(f"{self.func.__name__} took {end - start:.4f}s")
        return result

@Timer
def slow_function():
    time.sleep(0.1)
    return "Done"

slow_function()

In [ ]:
# Decorate a class
def singleton(cls):
    """Make class a singleton."""
    instances = {}
    
    @wraps(cls)
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    
    return get_instance

@singleton
class Database:
    def __init__(self):
        print("Creating database connection")
        self.connected = True

db1 = Database()
db2 = Database()
print(f"Same instance: {db1 is db2}")

In [ ]:
# Add methods to class
def add_repr(cls):
    """Add __repr__ method to class."""
    def __repr__(self):
        attrs = ', '.join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{cls.__name__}({attrs})"
    
    cls.__repr__ = __repr__
    return cls

@add_repr
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

p = Person("Alice", 25)
print(p)

## 7. Built-in Decorators

In [ ]:
# @property - getter/setter
class Circle:
    def __init__(self, radius):
        self._radius = radius
    
    @property
    def radius(self):
        return self._radius
    
    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("Radius must be positive")
        self._radius = value

c = Circle(5)
print(f"Radius: {c.radius}")
c.radius = 10
print(f"New radius: {c.radius}")

In [ ]:
# @staticmethod and @classmethod
class MyClass:
    class_var = "I'm a class variable"
    
    @staticmethod
    def static_method():
        return "Static method (no self or cls)"
    
    @classmethod
    def class_method(cls):
        return f"Class method: {cls.class_var}"

print(MyClass.static_method())
print(MyClass.class_method())

In [ ]:
# @functools.lru_cache
from functools import lru_cache

@lru_cache(maxsize=128)
def expensive_function(n):
    print(f"Computing for {n}")
    return n ** 2

print(expensive_function(5))
print(expensive_function(5))  # Cached!
print(expensive_function(10))

In [ ]:
# @dataclass (Python 3.7+)
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

p = Point(3, 4)
print(p)  # Auto-generated __repr__
print(p == Point(3, 4))  # Auto-generated __eq__

## Decorator Summary

| Pattern | Use Case |
|---------|----------|
| `@decorator` | Modify function behavior |
| `@decorator(args)` | Configurable decorator |
| `@wraps(func)` | Preserve function metadata |
| `@property` | Getter/setter |
| `@staticmethod` | No self/cls needed |
| `@classmethod` | Access class, not instance |
| `@lru_cache` | Memoization |
| `@dataclass` | Auto-generate class methods |